# Large Language Model Quantization with Quark

In this notebook we will explore how to quantize and evaluate a LLM using Quark.

## 🛠️ Supported Hardware

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
⚠️ AMD EPYC™ Processors  
⚠️ AMD Ryzen™ (AI) Processors  

Suggested hardware: **AMD Instinct™ Accelerators**, this notebook can run in a CPU as well but inference is CPU will be slow.

## ⚡ Recommended Software Environment

::::{tab-set}

:::{tab-item} Linux
- [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html)
- [Install PyTorch](https://amdresearch.github.io/aup-ai-tutorials//env/env-cpu.html)
:::

:::{tab-item} Windows
- [Install Direct-ML](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu-windows.html)
- [Install PyTorch](https://amdresearch.github.io/aup-ai-tutorials//env/env-cpu.html)
:::
::::

## 🎯 Goals

- Explore Quark to quantize a LLM
- Use LLM Eval to verify LLM performance

:::{seealso}
- [Quark](https://quark.docs.amd.com/latest/index.html)
- [Quantizing a Large Language Model with Quark](https://quark.docs.amd.com/latest/tutorials/torch/llm_tutorial/llm_tutorial.html)
- [LM Eval](https://zenodo.org/records/17728786)
- [Massive Multitask Language Understanding (MMLU)](https://en.wikipedia.org/wiki/MMLU)
:::

## Setup Environment 

Import packages and define device we will be using. It is recommended to use a GPU.

In [1]:
import os
import torch
import pygit2
from collections import deque

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using {device=}')

Using device=device(type='cuda')


## Get LLM Model

We will get the [mistralai/Mistral-7B-v0.1](https://huggingface.co/mistralai/Mistral-7B-v0.1) generative text model with 7 billion parameters. We download the model using the [hf Command Line Interface (CLI)](https://huggingface.co/docs/huggingface_hub/en/guides/cli). We assign the output of the download to a variable so we can get the download path.

In [3]:
model_path=!hf download mistralai/Mistral-7B-v0.1

Get model download path, we need this information to quantize the model.

In [4]:
MODEL_PATH = model_path[-1]
MODEL_PATH

'/root/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1/snapshots/27d67f1b5f57dc0953326b2601d68371d40ea8da'

## Clone the Quark Repository

We are going to clone the [Quark repository](https://github.com/amd/Quark) to use the LM Eval scripts. [LM Eval](https://github.com/EleutherAI/lm-evaluation-harness) provides a unified framework to test generative language models on a large number of different evaluation tasks.

In [5]:
pygit2.clone_repository('https://github.com/amd/Quark', 'Quark', checkout_branch='release/0.10')

pygit2.Repository('/shared-docker/Quark/.git/')

# Evaluate non Quantized Model

First of all, we are going to evaluate the original model. Let's start by defining filenames where we will store the result of the evaluation, so we can later compare the original and quantized model.

In [6]:
original_log_file = 'original_model_evaluation.txt'
quantized_log_file = 'quantized_model_evaluation.txt'

Using the Quark `llm_eval` script, we pass the path to the original mode, we define the [Massive Multitask Language Understanding (MMLU)](https://en.wikipedia.org/wiki/MMLU) tasks we want to evaluate and the batch size.

The tasks we will evaluate this model are:
- Professional Law
- Management
- Sociology
- Machine Learning

This process will take a few minutes, the result is a table with the four categories reporting the accuracy and standard error. The higher the accuracy the better. The idea is to compare the accuracy of the original model with the quantized one.

In [7]:
!python Quark/examples/torch/language_modeling/llm_eval/llm_eval.py \
    --model_args pretrained=$MODEL_PATH \
    --model hf \
    --tasks mmlu_professional_law,mmlu_management,mmlu_sociology,mmlu_machine_learning \
    --batch_size 1 \
    --device $device 2>&1 | tee $original_log_file

/opt/venv/lib/python3.10/site-packages/torchao/utils.py:408: UserWarning: TORCH_VERSION_AT_LEAST_2_5 is deprecated and will be removed in torchao 0.14.0
  warnings.warn(self.msg)
2025-12-24 15:14:56.824807990 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"

[QUARK-INFO]: C++ kernel compilation check start.

[QUARK-INFO]: C++ kernel build directory /root/.cache/torch_extensions/py310_cpu/kernel_ext

[QUARK-INFO]: C++ kernel loading. First-time compilation may take a few minutes...
Successfully preprocessed all matching files.

[QUARK-INFO]: C++ kernel compilation is already complete. Ending the C++ kernel compilation check. Total time: 0.3929 seconds
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.89s/it]
2025-12-24:15:15:06 INFO     [__main__:465] Selected Tasks: ['mmlu_professional_law', 'mmlu_management', 'mmlu_soc

## Quantize the model using Quark

Now, we can quantize the model using Quark. We are going to use the Post Training Quantization technique by running the `quantize_quark.py` script.

We need to provide the path to the original model, where we want to store the quantized model, the type of quantization scheme, the number of calibration samples, the quantization algorithm, the dataset for calibration, the sequence length, and the export format.

- Quantization scheme: Int4 Weight Only Quantization
- Quantization Algorithm: Activation-aware Weight Quantization (AWQ)

The quantization process will take a few minutes.

:::{note}
**AMD Quark** is a comprehensive cross-platform deep learning toolkit designed to simplify and enhance the quantization of deep learning models. Supporting both PyTorch and ONNX models, AMD Quark empowers developers to optimize their models for deployment on a wide range of hardware backends, achieving significant performance gains without compromising accuracy.
:::

In [8]:
!python3 Quark/examples/torch/language_modeling/llm_ptq/quantize_quark.py \
    --model_dir $MODEL_PATH \
    --output_dir output_dir \
    --quant_scheme w_int4_per_group_sym \
    --num_calib_data 128 \
    --quant_algo awq \
    --dataset pileval_for_awq_benchmark \
    --seq_len 512 \
    --model_export hf_format 2>&1 | tee quantization_log.txt

/opt/venv/lib/python3.10/site-packages/torchao/utils.py:408: UserWarning: TORCH_VERSION_AT_LEAST_2_5 is deprecated and will be removed in torchao 0.14.0
  warnings.warn(self.msg)

[QUARK-INFO]: C++ kernel compilation check start.

[QUARK-INFO]: C++ kernel build directory /root/.cache/torch_extensions/py310_cpu/kernel_ext

[QUARK-INFO]: C++ kernel loading. First-time compilation may take a few minutes...
Successfully preprocessed all matching files.

[QUARK-INFO]: C++ kernel compilation is already complete. Ending the C++ kernel compilation check. Total time: 0.3957 seconds
2025-12-24 15:16:06.683005395 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"

[INFO]: Loading model ...
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.86s/it]
Repo card metadata block was not found. Setting CardData to empty.

[QUARK-INFO]: Confi

Let's get the path to the quantized model

In [9]:
quant_model_path = os.path.join(os.getcwd(), 'output_dir')

## Evaluate Quantized Model

Similar as before, we are going to evaluate the quantized LLM with some MMLU tasks.

In [10]:
!python Quark/examples/torch/language_modeling/llm_eval/llm_eval.py \
    --model_args pretrained=$quant_model_path \
    --model hf \
    --tasks mmlu_professional_law,mmlu_management,mmlu_sociology,mmlu_machine_learning \
    --batch_size 1 \
    --device $device 2>&1 | tee $quantized_log_file

/opt/venv/lib/python3.10/site-packages/torchao/utils.py:408: UserWarning: TORCH_VERSION_AT_LEAST_2_5 is deprecated and will be removed in torchao 0.14.0
  warnings.warn(self.msg)
2025-12-24 15:23:51.972067654 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"

[QUARK-INFO]: C++ kernel compilation check start.

[QUARK-INFO]: C++ kernel build directory /root/.cache/torch_extensions/py310_cpu/kernel_ext

[QUARK-INFO]: C++ kernel loading. First-time compilation may take a few minutes...
Successfully preprocessed all matching files.

[QUARK-INFO]: C++ kernel compilation is already complete. Ending the C++ kernel compilation check. Total time: 0.4088 seconds
/opt/venv/lib/python3.10/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you'r

## Model Size

We now can verify that the quantization reduces the size significantly, a little bit over 7x.

In [11]:
def get_dir_size(path):
    """Get the size of a directory in Gigabytes."""
    size=!du -shLb $path
    return int(size[-1].split('\t')[0]) / (1024**3)

original_model_size=get_dir_size(MODEL_PATH)
quantized_model_size=get_dir_size(quant_model_path)

In [12]:
print(f'Original model size: {original_model_size:.2f} GB')
print(f'Quantized model size: {quantized_model_size:.2f} GB')
print(f'Size reduction: {original_model_size / quantized_model_size :.2f}x')

Original model size: 27.47 GB
Quantized model size: 3.87 GB
Size reduction: 7.10x


## Models Evaluation

Finally, we can compare the performance of the original and quantized model in the MMLU tasks.

In [13]:
def read_eval_table(path):
    """ Reads the table from the evaluation log file"""
    with open(path, 'r') as f:
        table = deque(f, maxlen=5)
    print(''.join(table))

In [14]:
read_eval_table(original_log_file)

|machine_learning|      1|none  |     0|acc   |↑  |0.4732|±  |0.0474|
|management      |      1|none  |     0|acc   |↑  |0.8058|±  |0.0392|
|professional_law|      1|none  |     0|acc   |↑  |0.4563|±  |0.0127|
|sociology       |      1|none  |     0|acc   |↑  |0.8657|±  |0.0241|




In [15]:
read_eval_table(quantized_log_file)

|machine_learning|      1|none  |     0|acc   |↑  |0.3482|±  |0.0452|
|management      |      1|none  |     0|acc   |↑  |0.8447|±  |0.0359|
|professional_law|      1|none  |     0|acc   |↑  |0.4205|±  |0.0126|
|sociology       |      1|none  |     0|acc   |↑  |0.8408|±  |0.0259|




In the Machine learning task, the quantized model lost 0.125 accuracy, ~35% worse than the original model. 
In Management, the quantized model won 0.04 accuracy, ~4.6% better than the original model.
In Professional Law, the quantized model lost 0.04 accuracy, ~8.5% worse than the original model.
In Sociology, the quantized model lost 0.02 accuracy, ~3% worse than the original model.

In general, the quantized model is less accurate. However, this is expected as we have done a significant quantization. Probably, with a better dataset, we could recover more accuracy on the ML task.

## Exercise for the Reader

- Perform a more extensive evaluation for more MMLU tasks. You can see a list of the tasks [here](https://huggingface.co/datasets/cais/mmlu#dataset-summary).

- Perform Quantization on a different LLM.

## Conclusion

In this notebook, we explored how to quantize an LLM using Quark. We analyzed the reduction in size after quantization. Finally, we used MMLU tasks to evaluate the accuracy loss of the quantized model. 

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT